# NASA Near Earth Orbit Analysis

## 1. Extract
Connect to the NASA API and pull the raw JSON.

In [17]:
#Import libraries 
import os
import requests
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from the root .env file
load_dotenv(dotenv_path="../.env") 

# Retrieve NASA api key
API_KEY = os.getenv("NASA_API_KEY")

# Build api url with the api key
# Build your URL using the environment variable
START_DATE = "2026-07-10"
END_DATE = "2026-07-17"
URL= f'https://api.nasa.gov/neo/rest/v1/feed?start_date={START_DATE}&end_date={END_DATE}&api_key={API_KEY}'

# Test the connection
# Fetch data
response = requests.get(URL)
raw_data = response.json()
print(f"Total elements returned: {raw_data['element_count']}")
print(f"Response Status: {response.status_code}")
print(raw_data['near_earth_objects'][START_DATE])  # Print the first NEO for the start date

Total elements returned: 35
Response Status: 200
[{'links': {'self': 'http://api.nasa.gov/neo/rest/v1/neo/2452639?api_key=opixCkmAWjzPYRwEwAXzjl6J7C1pcFQ5ED6CtZ89'}, 'id': '2452639', 'neo_reference_id': '2452639', 'name': '452639 (2005 UY6)', 'nasa_jpl_url': 'https://ssd.jpl.nasa.gov/tools/sbdb_lookup.html#/?sstr=2452639', 'absolute_magnitude_h': 18.14, 'estimated_diameter': {'kilometers': {'estimated_diameter_min': 0.6259720997, 'estimated_diameter_max': 1.3997161669}, 'meters': {'estimated_diameter_min': 625.9720996596, 'estimated_diameter_max': 1399.716166857}, 'miles': {'estimated_diameter_min': 0.3889609095, 'estimated_diameter_max': 0.8697430343}, 'feet': {'estimated_diameter_min': 2053.7143034471, 'estimated_diameter_max': 4592.2447888713}}, 'is_potentially_hazardous_asteroid': False, 'close_approach_data': [{'close_approach_date': '2026-07-10', 'close_approach_date_full': '2026-Jul-10 12:26', 'epoch_date_close_approach': 1783686360000, 'relative_velocity': {'kilometers_per_seco

In [ ]:
# Hold our flattened records
asteroid_list = []

# Loop through each date in the feed
neo_dates = raw_data['near_earth_objects']
for date in neo_dates:
    for asteroid in neo_dates[date]:
        
        # Extract close approach details
        close_approach = asteroid['close_approach_data'][0] if asteroid['close_approach_data'] else {}
        
        # Build a flat dictionary for each specific record
        asteroid_record = {
            'asteroid_id': asteroid['id'],
            'name': asteroid['name'],
            'absolute_magnitude_h': float(asteroid['absolute_magnitude_h']),
            'estimated_diameter_max_km': float(asteroid['estimated_diameter']['kilometers']['estimated_diameter_max']),
            'is_potentially_hazardous': bool(asteroid['is_potentially_hazardous_asteroid']),
            
            # Event specific details converted to floats from strings
            'close_approach_date': close_approach.get('close_approach_date'),
            'relative_velocity_kph': float(close_approach.get('relative_velocity', {}).get('kilometers_per_hour', 0)),
            'miss_distance_km': float(close_approach.get('miss_distance', {}).get('kilometers', 0)),
            'orbiting_body': close_approach.get('orbiting_body')
        }
        
        asteroid_list.append(asteroid_record)

df = pd.DataFrame(asteroid_list)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   asteroid_id                35 non-null     object 
 1   name                       35 non-null     object 
 2   absolute_magnitude_h       35 non-null     float64
 3   estimated_diameter_max_km  35 non-null     float64
 4   is_potentially_hazardous   35 non-null     bool   
 5   close_approach_date        35 non-null     object 
 6   relative_velocity_kph      35 non-null     float64
 7   miss_distance_km           35 non-null     float64
 8   orbiting_body              35 non-null     object 
dtypes: bool(1), float64(4), object(4)
memory usage: 2.3+ KB
Total objects reported by NASA: 35
Total rows in DataFrame: 35


## 2. Explore

Inspect the JSON structure and identify the fields we need.

## 3. Transform

Clean the data with pandas, handle missing values, convert types, engineer useful features.

## 4. Load to PostgreSQL

Export the cleaned dataset and import it into PostgreSQL.


## 5.Analyze

Write SQL queries to answer meaningful questions.


## 6. Tableau Dashboard

Build a Tableau dashboard around those insights.
